# 주간 차트 데이터 EDA
가설: "차트 포인트 합산 1위가 실제 '올해의 아티스트'나 '올해의 베스트송'을 받았는가?"  

분석: 연도별 차트 데이터 기반 예측 1위 vs 실제 수상자 매칭율 확인. 만약 다르다면 심사위원 점수나 투표가 어떤 변수가 되었는지 추론

### 심사대상
2024.10.31~2025.11.19 발매된 모든 음원  
각 부문별 음원 점수 집계  
단, 트랙제로 초이스 부문은 심사 기간 상이하고 핫트렌드 부문은 발매 시기 제약 없음

In [36]:
## 주간 차트 데이터 컬럼 정제
import pandas as pd

# 1. 2020~2025년 주간차트 데이터 불러오기
# 파일 리스트가 있다면 반복문으로 불러와서 한 번에 합치기
years = [2019, 2020, 2021, 2022, 2023, 2024, 2025]
df_list = [pd.read_csv(f'/Users/js/MMA/Melon_Music_Award/data/주간차트/melon_{y}.csv') for y in years]

# 리스트를 통째로 전달
df_total = pd.concat(df_list, ignore_index=True)
df_total.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32600 entries, 0 to 32599
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   year        32600 non-null  int64 
 1   week        32600 non-null  object
 2   week_start  32600 non-null  object
 3   week_end    32600 non-null  object
 4   rank        32600 non-null  int64 
 5   title       32600 non-null  object
 6   artist      32600 non-null  object
 7   album       32600 non-null  object
 8   like_cnt    32600 non-null  int64 
dtypes: int64(3), object(6)
memory usage: 2.2+ MB


In [199]:
df_total.head()

,year,week,week_start,week_end,rank,title,artist,album,like_cnt
0,2019,2019.09.30 ~ 2019.10.06,2019.09.30,2019.10.06,1,"어떻게 이별까지 사랑하겠어, 널 사랑하는 거지",AKMU (악뮤),항해,495320
1,2019,2019.09.30 ~ 2019.10.06,2019.09.30,2019.10.06,2,흔들리는 꽃들 속에서 네 샴푸향이 느껴진거야,장범준,멜로가 체질 OST Part 3,320118
2,2019,2019.09.30 ~ 2019.10.06,2019.09.30,2019.10.06,3,조금 취했어 (Prod. 2soo),임재현,조금 취했어,136247
3,2019,2019.09.30 ~ 2019.10.06,2019.09.30,2019.10.06,4,워커홀릭,볼빨간사춘기,Two Five,148140
4,2019,2019.09.30 ~ 2019.10.06,2019.09.30,2019.10.06,5,안녕,폴킴,호텔 델루나 OST Part.10,224259


In [200]:
# 중복 확인
df_total.duplicated().sum()

0

In [201]:
# 통계 확인
df_total['rank'].describe()

count    32600.000000
mean        50.500000
std         28.866513
min          1.000000
25%         25.750000
50%         50.500000
75%         75.250000
max        100.000000
Name: rank, dtype: float64

In [203]:
df_total.to_csv('/Users/js/MMA/Melon_Music_Award/data/melon_total.csv', index=False, encoding='utf-8-sig')

## 집계기간 별 데이터 나누기
###MMA 집계기간 정의  
2025 MMA 기준: 2024.10.31 ~ 2025.11.19  
→ 매년 전년도 10.31 ~ 당해년도 11.19 가정한다.

In [ ]:
import pandas as pd

# 1. 데이터 불러오기
df_total = pd.read_csv('/Users/js/MMA/Melon_Music_Award/data/주간차트/melon_total.csv')

# 2. 날짜 컬럼을 데이터타입으로 변환 (비교를 위해 필수)
df_total['week_start'] = pd.to_datetime(df_total['week_start'], format="mixed")
df_total['week_end'] = pd.to_datetime(df_total['week_end'], format='mixed')


In [27]:
# 3. 각 MMA 연도별 집계 기간 설정(2020~2022 집계기간은 가장 최근 2025의 집계기간으로 대체)
mma_periods = {
    "MMA_2020": ('2019-10-31', '2020-11-19'),
    "MMA_2021": ('2020-10-31', '2021-11-19'),
    "MMA_2022": ('2021-10-31', '2022-11-19'),
    "MMA_2023": ('2022-11-04', '2023-11-01'),
    "MMA_2024": ('2023-11-02', '2024-10-30'),
    "MMA_2025": ('2024-10-31', '2025-11-19')
}

In [ ]:
# 4. 기간별로 데이터 나누기
mma_datasets = {}

for mma_year, (start_date, end_date) in mma_periods.items():
    # 해당 기간에 포함되는 행만 필터링
    # 수정된 필터 조건: 종료일이 집계 시작일 이후이고, 시작일이 집계 종료일 이전인 경우
    mask = (df_total['week_end'] >= start_date) & (df_total['week_start'] <= end_date)
    mma_datasets[mma_year] = df_total.loc[mask].copy()
    
    # 잘 나눠졌는지 확인 출력
    print(f"{mma_year}: {len(mma_datasets[mma_year])}개의 행이 수집됨")


MMA_2020: 5600개의 행이 수집됨
MMA_2021: 5600개의 행이 수집됨
MMA_2022: 5600개의 행이 수집됨
MMA_2023: 5300개의 행이 수집됨
MMA_2024: 5300개의 행이 수집됨
MMA_2025: 5600개의 행이 수집됨


In [ ]:
# 5. 연도별 MMA 데이터만 따로 저장
for year, data in mma_datasets.items():
    data.to_csv(f'/Users/js/MMA/Melon_Music_Award/data/MMA 집계기간별 데이터/MMA_{year}.csv', index=False)

## 주간차트 랭크로 성적 변환

주요 지표 선정  
- 기본점수 : 100~1점  

- 보너스점수 : 
MMA 특성을 고려해서 멜론차트 1위한 곡은 영향력을 다른 곡보다 크게 반영  
(확실히 1위곡은 영향력이 커야함 하위권 곡은 1~10위 진입이 없고 차트에만 들어간 느낌  
1위 한번 하는게 하위권 차트인 하는것보다 어려움)  
==> 1위곡 보너스 100점, 2위 30점, 3위 10점, 4~10위 5점

### 음원점수가 반영되는 수상부문
| 부문 | 설명 | 선정 방식 | 음원 심사 여부 |
|---|---|---|---|
| 카카오뱅크 올해의 앨범 | 최고 인기 앨범 | 음원 60% + 심사 20% + 투표 20% | O |
| 올해의 아티스트 | 최고 인기 아티스트 | 음원 60% + 심사 20% + 투표 20% | O |
| 올해의 베스트송 | 최고 인기 곡 | 음원 60% + 심사 20% + 투표 20% | O |
| 올해의 신인 | 최고 인기 신인 | 음원 60% + 심사 20% + 투표 20% | O |
| 베스트 솔로 여자 | 최고 인기 솔로 여자 | 음원 60% + 심사 20% + 투표 20% | O |
| 베스트 솔로 남자 | 최고 인기 솔로 남자 | 음원 60% + 심사 20% + 투표 20% | O |
| 베스트 그룹 여자 | 최고 인기 그룹 여자 | 음원 60% + 심사 20% + 투표 20% | O |
| 베스트 그룹 남자 | 최고 인기 그룹 남자 | 음원 60% + 심사 20% + 투표 20% | O |
| 베스트 OST | 최고 인기 드라마/영화/웹툰 등 작품 삽입곡 | 음원 60% + 심사 20% + 투표 20% | O |
| 베스트 팝 아티스트 | 최고 인기 POP 아티스트 | 음원 60% + 심사 20% + 투표 20% | O |
| TOP10 | 가장 사랑 받은 아티스트 10팀 | 음원 60% + 심사 20% + 투표 20%  | O |
| 밀리언스 TOP10 | 발매 24시간 내 100만 스트리밍 이상 달성하여 멜론의 전당에 오른 앨범 | 음원 80% + 투표 20%| O |
| 트랙제로 초이스 | 자신만의 장르와 스타일을 드러내 대중문화의 다양성에 기여한 곡과 아티스트 *심사기간 : 2024.10.05~2025.10.04 | 음원 20% + 심사 60% + 투표 20% | O |

### **최종점수 산출 방식 음원 70% + 좋아요 30%**
심사기준은 보통 음원 60% + 심사 20% + 투표 20%를 반영  
=> 투표는 각 곡의 '좋아요'로 점수를 대체 / 심사는 알 수 없으므로 음원점수, 투표점수에 10%씩 추가하여 최종 점수 집계  
(밀리언스 TOP10 => 음원 80% + 투표 20% / 트랙제로 초이스는 심사의 비중이 너무큼 )

### 집계기간별 데이터로 EDA

In [173]:
# 2025년 MMA
import pandas as pd

df_2025 = pd.read_csv("/Users/js/MMA/Melon_Music_Award/data/MMA 집계기간별 데이터/MMA_2025.csv")
df_2025.head()
df_2024 = pd.read_csv("/Users/js/MMA/Melon_Music_Award/data/MMA 집계기간별 데이터/MMA_2024.csv")


### 1위 100점, 2위 20점, 3위 10점, 4~10위 1점반영

In [190]:
import numpy as np
def calculate_mma_score(df_mma):
    # 1. 곡별로 그룹화 (title과 artist를 기준으로)
    # like_cnt는 주차별로 변할 수 있으므로 최대값을 기준으
    song_groups = df_mma.groupby(['title', 'artist'])
    
    # 2. 지표 계산
    # 기본 점수: 101 - rank (1위 100점 ~ 100위 1점)
    df_mma['base_score'] = 101 - df_mma['rank']
    
    # 보너스 점수 계산 함수
    # 차트 1위는 2~100위 보다 강력한 점수 반영
    def get_bonus(rank):
        if rank == 1: return 100
        elif rank == 2: return 20
        elif rank == 3: return 10
        elif rank <= 10: return 1
        else: return 0
    
    df_mma['bonus_score'] = df_mma['rank'].apply(get_bonus)
    
    # 3. 곡별 최종 집계
    mma_final = song_groups.agg(
        album=('album', 'first'),
        total_base_score=('base_score', 'sum'),
        total_bonus_score=('bonus_score', 'sum'),
        chart_in_weeks=('rank', 'count'),      # 차트인 기간 (주차 수)
        top1_count=('rank', lambda x: (x == 1).sum()),  # Top 1 횟수
        top10_count=('rank', lambda x: (x <= 10).sum()), # Top 10 횟수
        max_likes=('like_cnt', 'max')         # 좋아요 수 (팬덤 지표)
    ).reset_index()
    
    # 4. 좋아요 수를 정규화
    mma_final['like_score'] = np.sqrt(mma_final['max_likes'])
    mma_final['like_score'] = (mma_final['like_score'] / mma_final['like_score'].max()) * 100
    
    # 5. 종합 점수 계산 (가중치는 필요에 따라 조정)
    # 5-1. 음원 점수(기본+보너스) 합산
    mma_final['raw_music_score'] = mma_final['total_base_score'] + mma_final['total_bonus_score']

    # # 5-2. 음원 점수를 0~100점으로 정규화 (1위 곡이 100점이 되도록)
    max_music = mma_final['raw_music_score'].max()
    mma_final['normalized_music_score'] = (mma_final['raw_music_score'] / max_music) * 100

    # 5-3. 최종 점수 산출 (음원 70% + 좋아요 30%)
    mma_final['total_score'] = (
        (mma_final['normalized_music_score'] * 0.7) + 
        (mma_final['like_score'] * 0.3)
    )

    # 순위 정렬
    mma_final = mma_final.sort_values(by='total_score', ascending=False)
    
    return mma_final

# 사용 예시 (2025 MMA 데이터 적용)
mma_2024_scored = calculate_mma_score(df_2024)


In [191]:
mma_2024_scored.head(20)

,title,artist,album,total_base_score,total_bonus_score,chart_in_weeks,top1_count,top10_count,max_likes,like_score,raw_music_score,normalized_music_score,total_score
265,헤어지자 말해요,박재정,1집 Alone,4287,16,53,0,16,193508,61.610006,4303,98.557032,87.472924
52,Hype Boy,NewJeans,NewJeans 1st EP 'New Jeans',3960,0,53,0,0,315188,78.629701,3960,90.700870,87.079520
53,I AM,IVE (아이브),I've IVE,4133,1,53,0,1,238147,68.347769,4134,94.686212,86.784679
145,그대만 있다면 (여름날 우리 X 너드커넥션 (Nerd Connection)),너드커넥션 (Nerd Connection),그대만 있다면 (여름날 우리 X 너드커넥션 (Nerd Connection)),4208,11,53,0,11,171102,57.933442,4219,96.633074,85.023184
161,너의 모든 순간,성시경,별에서 온 그대 OST Part.7,3746,0,53,0,0,322134,79.491386,3746,85.799359,83.906967
199,비의 랩소디,임재현,비의 랩소디,4025,341,48,3,17,109481,46.341615,4366,100.000000,83.902485
69,Love wins all,아이유,The Winning,3545,448,41,4,15,222390,66.047961,3993,91.456711,83.834086
92,Seven (feat. Latto) - Clean Ver.,정국,Seven (feat. Latto) - Clean Ver.,3946,9,53,0,9,230704,67.271227,3955,90.586349,83.591813
227,예뻤어,DAY6 (데이식스),Every DAY6 February,3588,4,48,0,4,380427,86.384824,3592,82.272103,83.505919
262,한 페이지가 될 수 있게,DAY6 (데이식스),The Book of Us : Gravity,3594,17,45,0,17,367301,84.881461,3611,82.707284,83.359537


In [193]:
years = ['2020', '2021', '2022', '2023', '2024', '2025']

mma_results = {}

# 2019~2025MMA 데이터 별 점수 추가 
for year in years:
    df = pd.read_csv(f"/Users/js/MMA/Melon_Music_Award/data/MMA 집계기간별 데이터/MMA_{year}.csv")
    scored_df = calculate_mma_score(df)
    mma_results[year] = scored_df

# 추가된 점수 저장
for year, df in mma_results.items():
    df.to_csv(f"/Users/js/MMA/Melon_Music_Award/data/MMA 집계기간별 데이터/MMA_scored_{year}_v1.csv", index=False)

### 기존 보너스 점수 계산 + 1위횟수한 만큼 추가 점수 반영

In [197]:
import numpy as np
def calculate_mma_score(df_mma):
    # 1. 곡별로 그룹화 (title과 artist를 기준으로)
    # like_cnt는 주차별로 변할 수 있으므로 최대값을 기준으
    song_groups = df_mma.groupby(['title', 'artist'])
    
    # 2. 지표 계산
    # 기본 점수: 101 - rank (1위 100점 ~ 100위 1점)
    df_mma['base_score'] = 101 - df_mma['rank']
    
    # 보너스 점수 계산 함수
    # 차트 1위는 2~100위 보다 강력한 점수 반영
    def get_bonus(rank):
        if rank == 1: return 100
        elif rank == 2: return 20
        elif rank == 3: return 10
        elif rank <= 10: return 1
        else: return 0
    
    df_mma['bonus_score'] = df_mma['rank'].apply(get_bonus)
    
    # 3. 곡별 최종 집계
    mma_final = song_groups.agg(
        album=('album', 'first'),
        total_base_score=('base_score', 'sum'),
        total_bonus_score=('bonus_score', 'sum'),
        chart_in_weeks=('rank', 'count'),      # 차트인 기간 (주차 수)
        top1_count=('rank', lambda x: (x == 1).sum()),  # Top 1 횟수
        top10_count=('rank', lambda x: (x <= 10).sum()), # Top 10 횟수
        max_likes=('like_cnt', 'max')         # 좋아요 수 (팬덤 지표)
    ).reset_index()
    
    # 4. 좋아요 수를 정규화
    mma_final['like_score'] = np.sqrt(mma_final['max_likes'])
    mma_final['like_score'] = (mma_final['like_score'] / mma_final['like_score'].max()) * 100
    
    # 5. 종합 점수 계산 (가중치는 필요에 따라 조정)
    # 5-1. 음원 점수(기본+보너스) 합산
    mma_final['raw_music_score'] = mma_final['total_base_score'] + mma_final['total_bonus_score']

    # # 5-2. 음원 점수를 0~100점으로 정규화 (1위 곡이 100점이 되도록)
    max_music = mma_final['raw_music_score'].max()
    mma_final['normalized_music_score'] = (mma_final['raw_music_score'] / max_music) * 100

    # 5-3. 최종 점수 산출 (음원 70% + 좋아요 30%)
    mma_final['total_score'] = (
        (mma_final['normalized_music_score'] * 0.7) + 
        (mma_final['like_score'] * 0.3)
    )
    # 5-4. 1위 횟수 만큼 추가 점수 반영
    mma_final['total_score'] += mma_final['top1_count'] * 1
    
    # 순위 정렬
    mma_final = mma_final.sort_values(by='total_score', ascending=False)
    
    return mma_final

# 사용 예시 (2025 MMA 데이터 적용)
mma_2025_scored = calculate_mma_score(df_2025)


In [198]:
mma_2025_scored.head(20)

,title,artist,album,total_base_score,total_bonus_score,chart_in_weeks,top1_count,top10_count,max_likes,like_score,raw_music_score,normalized_music_score,total_score
46,"HOME SWEET HOME (feat. 태양, 대성)",G-DRAGON,"HOME SWEET HOME (feat. 태양, 대성)",4896,1202,53,11,29,229386,67.078728,6098,100.000000,101.123618
29,Drowning,WOODZ,OO-LI,5227,571,56,2,46,295867,76.181519,5798,95.080354,91.410704
5,APT.,로제 (ROSÉ),APT.,4935,603,56,4,27,220555,65.774842,5538,90.816661,87.304116
120,Whiplash,aespa,Whiplash - The 5th Mini Album,5200,218,56,0,42,152272,54.652671,5418,88.848803,78.589963
207,"어떻게 이별까지 사랑하겠어, 널 사랑하는 거지",AKMU (악뮤),항해,4170,1,56,0,1,495297,98.567607,4171,68.399475,77.449915
247,한 페이지가 될 수 있게,DAY6 (데이식스),The Book of Us : Gravity,4450,0,56,0,0,367301,84.881377,4450,72.974746,76.546735
43,HAPPY,DAY6 (데이식스),Fourever,4872,17,56,0,17,213258,64.677620,4889,80.173827,75.524965
138,나는 반딧불,황가람,나는 반딧불,4982,59,56,0,32,158266,55.717956,5041,82.666448,74.581900
112,TOO BAD (feat. Anderson .Paak),G-DRAGON,Übermensch,3224,916,39,9,16,155640,55.253777,4140,67.891112,73.099911
213,예뻤어,DAY6 (데이식스),Every DAY6 February,4028,0,56,0,0,380427,86.384740,4028,66.054444,72.153533


In [199]:
years = ['2020', '2021', '2022', '2023', '2024', '2025']

mma_results = {}

# 2019~2025MMA 데이터 별 점수 추가 
for year in years:
    df = pd.read_csv(f"/Users/js/MMA/Melon_Music_Award/data/MMA 집계기간별 데이터/MMA_{year}.csv")
    scored_df = calculate_mma_score(df)
    mma_results[year] = scored_df

# 추가된 점수 저장
for year, df in mma_results.items():
    df.to_csv(f"/Users/js/MMA/Melon_Music_Award/data/MMA 집계기간별 데이터/MMA_scored_{year}_v2.csv", index=False)

# 적용한 점수를 토대로 곡,앨범,아티스트 점수 구하기

In [172]:
import pandas as pd

df_2025 = pd.read_csv('/Users/js/MMA/Melon_Music_Award/data/MMA 집계기간별 데이터/MMA_scored_2025_v2.csv')
df_2025.head()

,title,artist,album,total_base_score,total_bonus_score,chart_in_weeks,top1_count,top10_count,max_likes,like_score,raw_music_score,normalized_music_score,total_score
0,"HOME SWEET HOME (feat. 태양, 대성)",G-DRAGON,"HOME SWEET HOME (feat. 태양, 대성)",4896,1202,53,11,29,229386,67.078728,6098,100.000000,101.123618
1,Drowning,WOODZ,OO-LI,5227,571,56,2,46,295867,76.181519,5798,95.080354,91.410704
2,APT.,로제 (ROSÉ),APT.,4935,603,56,4,27,220555,65.774842,5538,90.816661,87.304116
3,Whiplash,aespa,Whiplash - The 5th Mini Album,5200,218,56,0,42,152272,54.652671,5418,88.848803,78.589963
4,"어떻게 이별까지 사랑하겠어, 널 사랑하는 거지",AKMU (악뮤),항해,4170,1,56,0,1,495297,98.567607,4171,68.399475,77.449915


In [173]:
artist_df = pd.read_csv('/Users/js/MMA/Melon_Music_Award/data/크롤링 데이터/artist_info_final.csv')
artist_df['artist'] = artist_df['아티스트명']
artist_df.head()

,아티스트명,국적,활동장르,데뷔,멤버,성별,구성,artist
0,로제 (ROSÉ),대한민국,"록/메탈, POP, 발라드",2021.03.12,NaN,여성,솔로,로제 (ROSÉ)
1,aespa,대한민국,"댄스, 일렉트로니카, R&B/Soul",2020.11.17,"카리나 (KARINA), 지젤 (GISELLE), 윈터 (WINTER), 닝닝 (N...",여성,그룹,aespa
2,제니 (JENNIE),대한민국,"댄스, 국외드라마, POP",2018.11.12,NaN,여성,솔로,제니 (JENNIE)
3,DAY6 (데이식스),대한민국,"록/메탈, 댄스, 발라드",2015.09.07,"성진 (DAY6), Young K (DAY6), 원필 (DAY6), 도운 (DAY6)",남성,그룹,DAY6 (데이식스)
4,G-DRAGON,대한민국,"댄스, 랩/힙합, 발라드",2009.08.18,NaN,남성,솔로,G-DRAGON


In [174]:
# 두 데이터 합치기
merged_2025 = df_2025.merge(
    artist_df[['artist','데뷔','성별','구성']],
    on='artist',
    how='left'
)

In [175]:
merged_2025.head()

,title,artist,album,total_base_score,total_bonus_score,chart_in_weeks,top1_count,top10_count,max_likes,like_score,raw_music_score,normalized_music_score,total_score,데뷔,성별,구성
0,"HOME SWEET HOME (feat. 태양, 대성)",G-DRAGON,"HOME SWEET HOME (feat. 태양, 대성)",4896,1202,53,11,29,229386,67.078728,6098,100.000000,101.123618,2009.08.18,남성,솔로
1,Drowning,WOODZ,OO-LI,5227,571,56,2,46,295867,76.181519,5798,95.080354,91.410704,2016.07.29,남성,솔로
2,APT.,로제 (ROSÉ),APT.,4935,603,56,4,27,220555,65.774842,5538,90.816661,87.304116,2021.03.12,여성,솔로
3,Whiplash,aespa,Whiplash - The 5th Mini Album,5200,218,56,0,42,152272,54.652671,5418,88.848803,78.589963,2020.11.17,여성,그룹
4,"어떻게 이별까지 사랑하겠어, 널 사랑하는 거지",AKMU (악뮤),항해,4170,1,56,0,1,495297,98.567607,4171,68.399475,77.449915,2014.04.07,혼성,그룹


### 곡 점수

In [58]:
# 2025 MMA 기간 음원별 점수
song_2025 = merged_2025.copy()
song_2025[['title','artist','total_score']].head()

,title,artist,total_score
0,"HOME SWEET HOME (feat. 태양, 대성)",G-DRAGON,101.123618
1,Drowning,WOODZ,91.410704
2,APT.,로제 (ROSÉ),87.304116
3,Whiplash,aespa,78.589963
4,"어떻게 이별까지 사랑하겠어, 널 사랑하는 거지",AKMU (악뮤),77.449915


### 아티스트 점수

In [59]:
artist_2025 = song_2025.groupby('artist').agg(
    artist_score=('total_score', 'sum'),
    song_count=('title', 'count')
)
artist_2025 = artist_2025.sort_values(by='artist_score', ascending=False).reset_index()

artist_2025.head()

,artist,artist_score,song_count
0,DAY6 (데이식스),410.163882,13
1,G-DRAGON,359.133845,10
2,aespa,322.881713,8
3,임영웅,287.308231,18
4,NewJeans,229.926558,7


### 앨범 점수

In [60]:
album_2025 = song_2025.groupby(['album', 'artist']).agg(
    album_score=('total_score', 'sum'),
    track_count=('title', 'count')
)
album_2025 = album_2025.sort_values(by='album_score', ascending=False).reset_index()

album_2025.head()

,album,artist,album_score,track_count
0,Fourever,DAY6 (데이식스),143.398214,2
1,Übermensch,G-DRAGON,141.663692,6
2,IVE EMPATHY,IVE (아이브),102.720940,2
3,"HOME SWEET HOME (feat. 태양, 대성)",G-DRAGON,101.123618,1
4,Armageddon - The 1st Album,aespa,97.903811,2


## 수상작 예측

In [61]:
# 올해의 베스트송
best_song = song_2025.sort_values('total_score', ascending=False).head(1)
best_song

,title,artist,album,total_base_score,total_bonus_score,chart_in_weeks,top1_count,top10_count,max_likes,like_score,raw_music_score,normalized_music_score,total_score,데뷔,성별,구성
0,"HOME SWEET HOME (feat. 태양, 대성)",G-DRAGON,"HOME SWEET HOME (feat. 태양, 대성)",4896,1202,53,11,29,229386,67.078728,6098,100.0,101.123618,2009.08.18,남성,솔로


In [62]:
# 올해의 아티스트
artist_award = artist_2025.sort_values('artist_score', ascending=False).head(1)
artist_award

,artist,artist_score,song_count
0,DAY6 (데이식스),410.163882,13


In [63]:
# 올해의 앨범
album_award = album_2025.sort_values('album_score', ascending=False).head(1)
album_award

,album,artist,album_score,track_count
0,Fourever,DAY6 (데이식스),143.398214,2


In [64]:
# 올해의 신인(2명)
target_year = 2025

rookie = artist_df[
    artist_df['데뷔'].str[:4].astype(int) == target_year
]

rookie_award = artist_2025[
    artist_2025['artist'].isin(rookie['아티스트명'])
].sort_values('artist_score', ascending=False).head(2)

rookie_award

,artist,artist_score,song_count
17,ALLDAY PROJECT,92.328227,3
34,조째즈,59.236083,1


In [65]:
# 베스트 솔로(여자)
female_solo_award = (
    merged_2025[
        (merged_2025['성별'] == '여성') &
        (merged_2025['구성'] == '솔로')
    ]
    .groupby('artist')['total_score']
    .sum()
    .reset_index()
    .sort_values('total_score', ascending=False)
)

female_solo_award.head()

,artist,total_score
6,아이유,183.466531
3,로제 (ROSÉ),174.139933
11,제니 (JENNIE),111.220155
9,이영지,43.480807
14,태연 (TAEYEON),37.945810


In [66]:
# 베스트 솔로(남자)
male_solo_award = (
    merged_2025[
        (merged_2025['성별'] == '남성') &
        (merged_2025['구성'] == '솔로')
    ]
    .groupby('artist')['total_score']
    .sum()
    .reset_index()
    .sort_values('total_score', ascending=False)
)

male_solo_award.head()

,artist,total_score
4,G-DRAGON,359.133845
27,임영웅,287.308231
24,이무진,179.494979
38,황가람,105.911982
5,WOODZ,98.143903


In [67]:
# 베스트 그룹(여자)
female_group_award = (
    merged_2025[
        (merged_2025['성별'] == '여성') &
        (merged_2025['구성'] == '그룹')
    ]
    .groupby('artist')['total_score']
    .sum()
    .reset_index()
    .sort_values('total_score',ascending=False)
)

female_group_award.head()

,artist,total_score
16,aespa,322.881713
12,NewJeans,229.926558
6,IVE (아이브),226.807227
13,QWER,136.617193
20,아일릿(ILLIT),91.795234


In [68]:
# 베스트 그룹(남자)
male_group_award = (
    merged_2025[
        (merged_2025['성별'] == '남성') &
        (merged_2025['구성'] == '그룹')
    ]
    .groupby('artist')['total_score']
    .sum()
    .reset_index()
    .sort_values('total_score',ascending=False)
)

male_group_award.head()

,artist,total_score
3,DAY6 (데이식스),410.163882
8,PLAVE,177.589102
1,BOYNEXTDOOR,162.010572
9,RIIZE,108.045080
11,TWS (투어스),89.737420


In [69]:
# TOP10
top10 = artist_2025.sort_values('artist_score', ascending=False).head(10)
top10

,artist,artist_score,song_count
0,DAY6 (데이식스),410.163882,13
1,G-DRAGON,359.133845,10
2,aespa,322.881713,8
3,임영웅,287.308231,18
4,NewJeans,229.926558,7
5,IVE (아이브),226.807227,6
6,아이유,183.466531,9
7,이무진,179.494979,4
8,PLAVE,177.589102,13
9,로제 (ROSÉ),174.139933,3


In [97]:
# 밀리언스 TOP10
millions = album_2025.sort_values('album_score', ascending=False).head(10)
millions

,album,artist,album_score,track_count
0,Fourever,DAY6 (데이식스),143.398214,2
1,Übermensch,G-DRAGON,141.663692,6
2,IVE EMPATHY,IVE (아이브),102.720940,2
3,"HOME SWEET HOME (feat. 태양, 대성)",G-DRAGON,101.123618,1
4,Armageddon - The 1st Album,aespa,97.903811,2
5,OO-LI,WOODZ,91.410704,1
6,APT.,로제 (ROSÉ),87.304116,1
7,IM HERO,임영웅,86.064079,5
8,꽃갈피 셋,아이유,83.849268,6
9,FAMOUS,ALLDAY PROJECT,80.889286,2


In [ ]:
rows = []

rows.append({
    '수상부문': '올해의 베스트송',
    '수상자': best_song.iloc[0]['title']
})

rows.append({
    '수상부문': '올해의 아티스트',
    '수상자': artist_award.iloc[0]['artist']
})

rows.append({
    '수상부문': '올해의 앨범',
    '수상자': album_award.iloc[0]['album']
})

rows.append({
    '수상부문': '올해의 신인',
    '수상자': rookie_award.iloc[0]['artist']
})
rows.append({
    '수상부문': '올해의 신인',
    '수상자': rookie_award.iloc[1]['artist']
})


rows.append({
    '수상부문': '베스트 솔로 여자',
    '수상자': female_solo_award.iloc[0]['artist']
})

rows.append({
    '수상부문': '베스트 솔로 남자',
    '수상자': male_solo_award.iloc[0]['artist']
})

rows.append({
    '수상부문': '베스트 그룹 여자',
    '수상자': female_group_award.iloc[0]['artist']
})


rows.append({
    '수상부문': '베스트 그룹 남자',
    '수상자': male_group_award.iloc[0]['artist']
})

# top10
for i in range(10):
    rows.append({
        '수상부문':'TOP10',
        '수상자': top10['artist'][i]
    })
    
for i in range(10):
    rows.append({
        '수상부문':'밀리언스 TOP10',
        '수상자': millions['album'][i]
    })

pre_award_2025 = pd.DataFrame(rows)

### 예측한 수상자 명단

In [123]:
pre_award_2025

,수상부문,수상자
0,올해의 베스트송,"HOME SWEET HOME (feat. 태양, 대성)"
1,올해의 아티스트,DAY6 (데이식스)
2,올해의 앨범,Fourever
3,올해의 신인,ALLDAY PROJECT
4,올해의 신인,조째즈
5,베스트 솔로 여자,아이유
6,베스트 솔로 남자,G-DRAGON
7,베스트 그룹 여자,aespa
8,베스트 그룹 남자,DAY6 (데이식스)
9,TOP10,DAY6 (데이식스)


In [124]:
# 실제 수상 명단
real_award_2025 = pd.read_csv('/Users/js/MMA/Melon_Music_Award/data/MMA_수상내역/2025_MMA_수상내역.csv')
real_award_2025['수상부문'] = real_award_2025['category']

In [128]:
real_award_2025.columns

Index(['부문 구분', 'category', '아티스트', '곡명', '앨범', '이미지', '수상부문'], dtype='object')

In [129]:
pre_award_2025.columns

Index(['수상부문', '수상자'], dtype='object')

In [222]:
def check_match(row):
    cond = real_award_2025['수상부문'] == row['수상부문']
    
    if row['수상부문'] in ['올해의 앨범', '밀리언스 TOP10']:
        return (cond & (real_award_2025['앨범'] == row['수상자'])).any()
    
    elif row['수상부문'] == '올해의 베스트송':
        return (cond & (real_award_2025['곡명'] == row['수상자'])).any()
    
    else:
        return (cond & (real_award_2025['아티스트'] == row['수상자'])).any()


compare_df = pre_award_2025.copy()
compare_df['정답여부'] = compare_df.apply(check_match, axis=1)

In [223]:
compare_df

,수상부문,수상자,정답여부
0,올해의 베스트송,"HOME SWEET HOME (feat. 태양, 대성)",True
1,올해의 아티스트,DAY6 (데이식스),False
2,올해의 앨범,Fourever,False
3,올해의 신인,ALLDAY PROJECT,True
4,올해의 신인,조째즈,False
5,베스트 솔로 여자,아이유,False
6,베스트 솔로 남자,G-DRAGON,True
7,베스트 그룹 여자,aespa,False
8,베스트 그룹 남자,DAY6 (데이식스),False
9,TOP10,DAY6 (데이식스),False


In [132]:
accuracy = compare_df['정답여부'].mean()
print(f"정확도: {accuracy*100:.1f}%")


정확도: 41.4%


### 29개 수상부문 중 12개 예측 성공

## 제대로 예측하지 못한 이유

In [133]:
# 예측 수상자 명단을 확인한 결과 주간차트는 MMA 집계기간으로 나누어 수집했지만
# 주간차트 내의 음원도 집계기간 내에 발매된 음원을 평가해야함

### 발매일 정보 추가

In [176]:
### 발매일 정보 추가
album_data = pd.read_csv('/Users/js/MMA/Melon_Music_Award/data/크롤링 데이터/album_data.csv')
album_data

,artist,album,release_date
0,AKMU (악뮤),항해,2019.09.25
1,장범준,멜로가 체질 OST Part 3,2019.08.23
2,임재현,조금 취했어,2019.09.24
3,볼빨간사춘기,Two Five,2019.09.10
4,폴킴,호텔 델루나 OST Part.10,2019.08.12
...,...,...,...
1309,Crush,스트릿 우먼 파이터2(SWF2) 계급미션,2023.09.05
1310,미연 (MIYEON),스트릿 우먼 파이터2(SWF2) 계급미션,2023.09.05
1311,폴킴,'키스 먼저 할까요?' OST Part.3,2018.03.20
1312,2PM,MUST,2021.06.28


In [177]:
### 차트 + 아티스트 데이터 + 발매일 데이터 병합
merged_2025 = merged_2025.merge(
    album_data[['artist', 'album', 'release_date']],
    on=['artist','album'],
    how='left'
)

In [178]:
merged_2025.head()

,title,artist,album,total_base_score,total_bonus_score,chart_in_weeks,top1_count,top10_count,max_likes,like_score,raw_music_score,normalized_music_score,total_score,데뷔,성별,구성,release_date
0,"HOME SWEET HOME (feat. 태양, 대성)",G-DRAGON,"HOME SWEET HOME (feat. 태양, 대성)",4896,1202,53,11,29,229386,67.078728,6098,100.000000,101.123618,2009.08.18,남성,솔로,2024.11.22
1,Drowning,WOODZ,OO-LI,5227,571,56,2,46,295867,76.181519,5798,95.080354,91.410704,2016.07.29,남성,솔로,2023.04.26
2,APT.,로제 (ROSÉ),APT.,4935,603,56,4,27,220555,65.774842,5538,90.816661,87.304116,2021.03.12,여성,솔로,2024.10.18
3,Whiplash,aespa,Whiplash - The 5th Mini Album,5200,218,56,0,42,152272,54.652671,5418,88.848803,78.589963,2020.11.17,여성,그룹,2024.10.21
4,"어떻게 이별까지 사랑하겠어, 널 사랑하는 거지",AKMU (악뮤),항해,4170,1,56,0,1,495297,98.567607,4171,68.399475,77.449915,2014.04.07,혼성,그룹,2019.09.25


In [179]:
# 집계기간
mma_periods = {
    "MMA_2020": ('2019-10-31', '2020-11-19'),
    "MMA_2021": ('2020-10-31', '2021-11-19'),
    "MMA_2022": ('2021-10-31', '2022-11-19'),
    "MMA_2023": ('2022-11-04', '2023-11-01'),
    "MMA_2024": ('2023-11-02', '2024-10-30'),
    "MMA_2025": ('2024-10-31', '2025-11-19')
}

In [180]:
# 1. release_date 컬럼을 datetime 형식으로 변환
# 데이터 형태에 따라 오류가 날 수 있으므로 errors='coerce'를 사용
merged_2025['release_date'] = pd.to_datetime(merged_2025['release_date'], errors='coerce')

# 2. MMA_2025 기준 날짜 가져오기
start_date, end_date = mma_periods["MMA_2025"]

# 3. 해당 기간에 포함되는 데이터만 필터링
# 발매일이 시작일과 종료일 사이에 있는 데이터만 남깁니다.
final_2025 = merged_2025[
    (merged_2025['release_date'] >= start_date) & 
    (merged_2025['release_date'] <= end_date)
].reset_index().copy()

# 4. 결과 확인
print(f"필터링 전: {len(merged_2025)}곡")
print(f"필터링 후: {len(final_2025)}곡")
print(final_2025[['title', 'artist', 'release_date']].head())

필터링 전: 253곡
필터링 후: 130곡
                            title     artist release_date
0  HOME SWEET HOME (feat. 태양, 대성)   G-DRAGON   2024-11-22
1  TOO BAD (feat. Anderson .Paak)   G-DRAGON   2025-02-25
2                         너에게 닿기를       10CM   2025-03-06
3              toxic till the end  로제 (ROSÉ)   2024-12-06
4                     REBEL HEART  IVE (아이브)   2025-02-03


In [257]:
final_2025.loc[final_2025['artist']=='조째즈']

,index,title,artist,album,total_base_score,total_bonus_score,chart_in_weeks,top1_count,top10_count,max_likes,like_score,raw_music_score,normalized_music_score,total_score,데뷔,성별,구성,release_date
7,26,모르시나요(PROD.로코베리),조째즈,모르시나요,3790,175,41,0,34,106644,45.7372,3965,65.021318,59.236083,2025.01.07,남성,솔로,2025-01-07


In [262]:
final_2025.loc[final_2025['artist']=='Hearts2Hearts (하츠투하츠)']

,index,title,artist,album,total_base_score,total_bonus_score,chart_in_weeks,top1_count,top10_count,max_likes,like_score,raw_music_score,normalized_music_score,total_score,데뷔,성별,구성,release_date
30,89,STYLE,Hearts2Hearts (하츠투하츠),STYLE,1336,0,22,0,0,58678,33.926497,1336,21.908823,25.514125,2025.02.24,여성,그룹,2025-06-18
41,113,The Chase,Hearts2Hearts (하츠투하츠),The Chase,999,0,27,0,0,50354,31.428116,999,16.382420,20.896129,2025.02.24,여성,그룹,2025-02-24
73,185,FOCUS,Hearts2Hearts (하츠투하츠),FOCUS - The 1st Mini Album,235,0,5,0,0,41569,28.555262,235,3.853723,11.264184,2025.02.24,여성,그룹,2025-10-20


### 점수계산

In [181]:
# 2025 MMA 기간 음원별 점수
# 곡 점수
song_2025 = final_2025.copy()
song_2025[['title','artist','total_score']].head()

,title,artist,total_score
0,"HOME SWEET HOME (feat. 태양, 대성)",G-DRAGON,101.123618
1,TOO BAD (feat. Anderson .Paak),G-DRAGON,73.099911
2,너에게 닿기를,10CM,66.797671
3,toxic till the end,로제 (ROSÉ),62.674983
4,REBEL HEART,IVE (아이브),62.293232


In [182]:
# 아티스트 점수
artist_2025 = song_2025.groupby('artist').agg(
    artist_score=('total_score', 'sum'),
    song_count=('title', 'count')
)
artist_2025 = artist_2025.sort_values(by='artist_score', ascending=False).reset_index()

artist_2025.head()

,artist,artist_score,song_count
0,G-DRAGON,315.971974,9
1,BOYNEXTDOOR,162.010572,10
2,임영웅,134.527177,10
3,IVE (아이브),126.454395,3
4,아이유,101.827919,7


In [184]:
# 앨범 점수
album_2025 = song_2025.groupby(['album', 'artist']).agg(
    album_score=('total_score', 'sum'),
    track_count=('title', 'count')
)
album_2025 = album_2025.sort_values(by='album_score', ascending=False).reset_index()

album_2025.head()

,album,artist,album_score,track_count
0,Übermensch,G-DRAGON,141.663692,6
1,IVE EMPATHY,IVE (아이브),102.720940,2
2,"HOME SWEET HOME (feat. 태양, 대성)",G-DRAGON,101.123618,1
3,IM HERO,임영웅,86.064079,5
4,꽃갈피 셋,아이유,83.849268,6


### 수상 예측

In [185]:
# 올해의 베스트송
best_song = song_2025.sort_values('total_score', ascending=False).head(1)
best_song

,index,title,artist,album,total_base_score,total_bonus_score,chart_in_weeks,top1_count,top10_count,max_likes,like_score,raw_music_score,normalized_music_score,total_score,데뷔,성별,구성,release_date
0,0,"HOME SWEET HOME (feat. 태양, 대성)",G-DRAGON,"HOME SWEET HOME (feat. 태양, 대성)",4896,1202,53,11,29,229386,67.078728,6098,100.0,101.123618,2009.08.18,남성,솔로,2024-11-22


In [186]:
# 올해의 아티스트
artist_award = artist_2025.sort_values('artist_score', ascending=False).head(1)
artist_award

,artist,artist_score,song_count
0,G-DRAGON,315.971974,9


In [187]:
# 올해의 앨범
album_award = album_2025.sort_values('album_score', ascending=False).head(1)
album_award

,album,artist,album_score,track_count
0,Übermensch,G-DRAGON,141.663692,6


In [242]:
# 올해의 신인(2명)
target_year = 2025

rookie = final_2025[
    pd.to_datetime(final_2025['데뷔'], errors='coerce').dt.year == target_year
]

rookie_award = artist_2025[
    artist_2025['artist'].isin(rookie['artist'])
].sort_values('artist_score', ascending=False).head(3)

rookie_award

,artist,artist_score,song_count
6,ALLDAY PROJECT,80.889286,2
11,조째즈,59.236083,1
12,Hearts2Hearts (하츠투하츠),57.674438,3


In [210]:
# 베스트 솔로(여자)
female_solo_award = (
    final_2025[
        (final_2025['성별'] == '여성') &
        (final_2025['구성'] == '솔로')
    ]
    .groupby('artist')['total_score']
    .sum()
    .reset_index()
    .sort_values('total_score', ascending=False)
)

female_solo_award.head(3)

,artist,total_score
4,아이유,101.827919
2,로제 (ROSÉ),86.835818
7,제니 (JENNIE),72.798196


In [211]:
# 베스트 솔로(남자)
male_solo_award = (
    final_2025[
        (final_2025['성별'] == '남성') &
        (final_2025['구성'] == '솔로')
    ]
    .groupby('artist')['total_score']
    .sum()
    .reset_index()
    .sort_values('total_score', ascending=False)
)

male_solo_award.head(3)

,artist,total_score
3,G-DRAGON,315.971974
18,임영웅,134.527177
0,10CM,66.797671


In [214]:
# 베스트 그룹(여자)
female_group_award = (
    final_2025[
        (final_2025['성별'] == '여성') &
        (final_2025['구성'] == '그룹')
    ]
    .groupby('artist')['total_score']
    .sum()
    .reset_index()
    .sort_values('total_score',ascending=False)
)

female_group_award.head(3)

,artist,total_score
5,IVE (아이브),126.454395
4,Hearts2Hearts (하츠투하츠),57.674438
11,aespa,55.741694


In [215]:
# 베스트 그룹(남자)
male_group_award = (
    final_2025[
        (final_2025['성별'] == '남성') &
        (final_2025['구성'] == '그룹')
    ]
    .groupby('artist')['total_score']
    .sum()
    .reset_index()
    .sort_values('total_score',ascending=False)
)

male_group_award.head(3)

,artist,total_score
0,BOYNEXTDOOR,162.010572
5,PLAVE,68.651216
2,DAY6 (데이식스),61.907205


In [249]:
# TOP10
top10 = artist_2025.sort_values('artist_score', ascending=False).head(20)
top10

,artist,artist_score,song_count
0,G-DRAGON,315.971974,9
1,BOYNEXTDOOR,162.010572,10
2,임영웅,134.527177,10
3,IVE (아이브),126.454395,3
4,아이유,101.827919,7
5,로제 (ROSÉ),86.835818,2
6,ALLDAY PROJECT,80.889286,2
7,제니 (JENNIE),72.798196,3
8,PLAVE,68.651216,5
9,10CM,66.797671,1


In [248]:
# 밀리언스 TOP10
millions = album_2025.sort_values('album_score', ascending=False).head(10)
millions

,album,artist,album_score,track_count
0,Übermensch,G-DRAGON,141.663692,6
1,IVE EMPATHY,IVE (아이브),102.720940,2
2,"HOME SWEET HOME (feat. 태양, 대성)",G-DRAGON,101.123618,1
3,IM HERO,임영웅,86.064079,5
4,꽃갈피 셋,아이유,83.849268,6
5,FAMOUS,ALLDAY PROJECT,80.889286,2
6,Caligo Pt.1,PLAVE,68.651216,5
7,너에게 닿기를,10CM,66.797671,1
8,rosie,로제 (ROSÉ),62.674983,1
9,오늘만 I LOVE YOU,BOYNEXTDOOR,61.367819,1


### 다시 예측한 수상자 명단

In [218]:
rows2 = []

rows2.append({
    '수상부문': '올해의 베스트송',
    '수상자': best_song.iloc[0]['title']
})

rows2.append({
    '수상부문': '올해의 아티스트',
    '수상자': artist_award.iloc[0]['artist']
})

rows2.append({
    '수상부문': '올해의 앨범',
    '수상자': album_award.iloc[0]['album']
})

rows2.append({
    '수상부문': '올해의 신인',
    '수상자': rookie_award.iloc[0]['artist']
})
rows2.append({
    '수상부문': '올해의 신인',
    '수상자': rookie_award.iloc[1]['artist']
})


rows2.append({
    '수상부문': '베스트 솔로 여자',
    '수상자': female_solo_award.iloc[0]['artist']
})

rows2.append({
    '수상부문': '베스트 솔로 남자',
    '수상자': male_solo_award.iloc[0]['artist']
})

rows2.append({
    '수상부문': '베스트 그룹 여자',
    '수상자': female_group_award.iloc[0]['artist']
})


rows2.append({
    '수상부문': '베스트 그룹 남자',
    '수상자': male_group_award.iloc[0]['artist']
})

# top10
for i in range(10):
    rows2.append({
        '수상부문':'TOP10',
        '수상자': top10['artist'][i]
    })
    
for i in range(10):
    rows2.append({
        '수상부문':'밀리언스 TOP10',
        '수상자': millions['album'][i]
    })

pre2_award_2025 = pd.DataFrame(rows2)

In [219]:
pre2_award_2025

,수상부문,수상자
0,올해의 베스트송,"HOME SWEET HOME (feat. 태양, 대성)"
1,올해의 아티스트,G-DRAGON
2,올해의 앨범,Übermensch
3,올해의 신인,ALLDAY PROJECT
4,올해의 신인,조째즈
5,베스트 솔로 여자,아이유
6,베스트 솔로 남자,G-DRAGON
7,베스트 그룹 여자,IVE (아이브)
8,베스트 그룹 남자,BOYNEXTDOOR
9,TOP10,G-DRAGON


In [224]:
def check_match(row):
    cond = real_award_2025['수상부문'] == row['수상부문']
    
    if row['수상부문'] in ['올해의 앨범', '밀리언스 TOP10']:
        return (cond & (real_award_2025['앨범'] == row['수상자'])).any()
    
    elif row['수상부문'] == '올해의 베스트송':
        return (cond & (real_award_2025['곡명'] == row['수상자'])).any()
    
    else:
        return (cond & (real_award_2025['아티스트'] == row['수상자'])).any()


compare_df2 = pre2_award_2025.copy()
compare_df2['정답여부'] = compare_df2.apply(check_match, axis=1)

In [225]:
compare_df2

,수상부문,수상자,정답여부
0,올해의 베스트송,"HOME SWEET HOME (feat. 태양, 대성)",True
1,올해의 아티스트,G-DRAGON,True
2,올해의 앨범,Übermensch,True
3,올해의 신인,ALLDAY PROJECT,True
4,올해의 신인,조째즈,False
5,베스트 솔로 여자,아이유,False
6,베스트 솔로 남자,G-DRAGON,True
7,베스트 그룹 여자,IVE (아이브),True
8,베스트 그룹 남자,BOYNEXTDOOR,True
9,TOP10,G-DRAGON,True


In [226]:
accuracy = compare_df2['정답여부'].mean()
print(f"정확도: {accuracy*100:.1f}%")


정확도: 65.5%


### 다시 시도한 결과
예측한 29개 부문 중 정답 개수 12개(41.4%) => 19개(65.5%)

In [228]:
compare_df.to_csv('예측결과저장.csv')
compare_df2.to_csv('예측결과저장2.csv')

# 틀린 10개 부문, 맞춘 부문 분석

In [438]:
real_award_2025[['수상부문','앨범']]

,수상부문,앨범
0,올해의 앨범,Übermensch
1,올해의 아티스트,NaN
2,올해의 베스트송,NaN
3,올해의 레코드,like JENNIE
4,올해의 신인,NaN
5,올해의 신인,NaN
6,베스트 솔로 여자,NaN
7,베스트 솔로 남자,NaN
8,베스트 그룹 여자,NaN
9,베스트 그룹 남자,NaN


In [239]:
compare_df2

,수상부문,수상자,정답여부
0,올해의 베스트송,"HOME SWEET HOME (feat. 태양, 대성)",True
1,올해의 아티스트,G-DRAGON,True
2,올해의 앨범,Übermensch,True
3,올해의 신인,ALLDAY PROJECT,True
4,올해의 신인,조째즈,False
5,베스트 솔로 여자,아이유,False
6,베스트 솔로 남자,G-DRAGON,True
7,베스트 그룹 여자,IVE (아이브),True
8,베스트 그룹 남자,BOYNEXTDOOR,True
9,TOP10,G-DRAGON,True


In [235]:
wrong1 = compare_df[compare_df['정답여부'] == False]
wrong1

,수상부문,수상자,정답여부
1,올해의 아티스트,DAY6 (데이식스),False
2,올해의 앨범,Fourever,False
4,올해의 신인,조째즈,False
5,베스트 솔로 여자,아이유,False
7,베스트 그룹 여자,aespa,False
8,베스트 그룹 남자,DAY6 (데이식스),False
9,TOP10,DAY6 (데이식스),False
13,TOP10,NewJeans,False
15,TOP10,아이유,False
16,TOP10,이무진,False


In [236]:
wrong2 = compare_df2[compare_df2['정답여부'] == False]
wrong2

,수상부문,수상자,정답여부
4,올해의 신인,조째즈,False
5,베스트 솔로 여자,아이유,False
13,TOP10,아이유,False
15,TOP10,ALLDAY PROJECT,False
18,TOP10,10CM,False
21,밀리언스 TOP10,"HOME SWEET HOME (feat. 태양, 대성)",False
22,밀리언스 TOP10,IM HERO,False
24,밀리언스 TOP10,FAMOUS,False
26,밀리언스 TOP10,너에게 닿기를,False
28,밀리언스 TOP10,오늘만 I LOVE YOU,False


### 계속 틀리는 부문
올해의 신인 >> 
- 예측 : 조째즈 
- 정답 : Hearts2Hearts(하츠투하츠) 

베스트 솔로 여자 >>
- 예측 : 아이유
- 정답 : 로제 (ROSÉ)  

TOP10 >>
- 예측 : **아이유** / **ALLDAY PROJECT** / **10CM** / G-DRAGON / 로제 (ROSÉ) / 임영웅 / 제니 (JENNIE) / BOYNEXTDOOR / IVE (아이브) / PLAVE
- 정답 : **RIIZE** / **aespa** / **NCT WISH** / 로제 (ROSÉ) / 임영웅 / 제니 (JENNIE) / BOYNEXTDOOR / G-DRAGON / IVE (아이브) / PLAVE

밀리언스 TOP10 >>
- 예측 : Übermensch / IVE EMPATHY / HOME SWEET HOME (feat. 태양, 대성) / IM HERO / 꽃갈피 셋 / FAMOUS / Caligo Pt.1 / 너에게 닿기를 / rosie / 오늘만 I LOVE YOU	
- 정답 : 
꽃갈피 셋 / Caligo Pt.1 / IM HERO 2 / IVE EMPATHY / No Genre / ODYSSEY - The 1st Album / rosie / Ruby / SEVENTEEN 5th Album ‘HAPPY BURSTDAY’ / Übermensch



In [233]:
compare_df2.groupby('수상부문')['정답여부'].mean()

수상부문
TOP10         0.7
밀리언스 TOP10    0.5
베스트 그룹 남자     1.0
베스트 그룹 여자     1.0
베스트 솔로 남자     1.0
베스트 솔로 여자     0.0
올해의 베스트송      1.0
올해의 신인        0.5
올해의 아티스트      1.0
올해의 앨범        1.0
Name: 정답여부, dtype: float64

### 올해의 신인, 베스트 솔로 여자, top10 ,밀리언스top10 부문 틀림  

- 올해의 신인 :   
ALLDAY PROJECT 80점 / 조째즈 59점 / Hearts2Hearts (하츠투하츠) 57점  => 조쨰즈, Hearts2Hearts 2점 차이  
- 베스트 솔로 여자 :  
1위 아이유 101점, 2위 로제 86점 => 약 15점 차이  
- top10 :  
14위 aespa 55점, 16위 RIIZE 50점, 32위 NCT WISH 25점 =>  
aespa, RIIZE, NCT WISH 세 그룹은 음원 점수 보다 심사, 투표점수에서 높은 점수를 획득했을 것으로 예측됨
- 밀리언스 top10 :  
예측 성공: Übermensch, IVE EMPATHY, 꽃갈피 셋, Caligo Pt.1, rosie  
예측 실패: HOME SWEET HOME (feat. 태양, 대성), IM HERO, FAMOUS, 너에게 닿기를, 오늘만 I LOVE YOU  
==> 실패한 정답: No Genre / ODYSSEY - The 1st Album / Ruby / SEVENTEEN 5th Album ‘HAPPY BURSTDAY’ / IM HERO 2  
** 밀리언스 top10의 후보 선정기준 확인

In [ ]:
# 로제 (ROSÉ) : toxic till the end, number one girl 2곡으로 86점
# 아이유 : 7곡으로 101점

# 조째즈 : 1곡으로 59점
# Hearts2Hearts (하츠투하츠) : 3곡으로 56점

### 전체 기간 (2020~2025)MMA 결과 예측 및 분석 진행

In [318]:
df_2025 = pd.read_csv('/Users/js/MMA/Melon_Music_Award/data/MMA 집계기간별 데이터/MMA_scored_2025_v2.csv')
df_2024 = pd.read_csv('/Users/js/MMA/Melon_Music_Award/data/MMA 집계기간별 데이터/MMA_scored_2024_v2.csv')
df_2023 = pd.read_csv('/Users/js/MMA/Melon_Music_Award/data/MMA 집계기간별 데이터/MMA_scored_2023_v2.csv')
df_2022 = pd.read_csv('/Users/js/MMA/Melon_Music_Award/data/MMA 집계기간별 데이터/MMA_scored_2022_v2.csv')
df_2021 = pd.read_csv('/Users/js/MMA/Melon_Music_Award/data/MMA 집계기간별 데이터/MMA_scored_2021_v2.csv')
df_2020 = pd.read_csv('/Users/js/MMA/Melon_Music_Award/data/MMA 집계기간별 데이터/MMA_scored_2020_v2.csv')

In [319]:
artist_df = pd.read_csv('/Users/js/MMA/Melon_Music_Award/data/크롤링 데이터/artist_info_final.csv')
artist_df['artist'] = artist_df['아티스트명']

In [320]:
# 발매일 정보
album_data = pd.read_csv('/Users/js/MMA/Melon_Music_Award/data/크롤링 데이터/album_data.csv')

In [321]:
# 차트 + 아티스트 데이터 + 발매일 데이터 병합
merged_df = {} # 결과를 담을 빈 딕셔너리

for i in range(2020, 2026):
    merged_df[i] = globals()[f'df_{i}'].merge(
        artist_df[['artist', '데뷔', '성별', '구성']], 
        on='artist', 
        how='left'
    ).merge(
        album_data[['artist', 'album', 'release_date']], 
        on=['artist', 'album'], 
        how='left'
    )

# 확인 예시: 2020년 데이터만 보고 싶을 때
# print(merged_df[2020].head())


In [322]:
for i in range(2020, 2026):
    merged_df[i]['release_date'] = pd.to_datetime(
        merged_df[i]['release_date'], errors='coerce'
    )
    merged_df[i]['데뷔'] = pd.to_datetime(
        merged_df[i]['데뷔'], errors='coerce'
    )

In [340]:
merged_df[2020]

,title,artist,album,total_base_score,total_bonus_score,chart_in_weeks,top1_count,top10_count,max_likes,like_score,raw_music_score,normalized_music_score,total_score,데뷔,성별,구성,release_date
0,Blueming,아이유,Love poem,4569,343,53,3,18,369754,85.164176,4912,100.000000,98.549253,2008-09-18,여성,솔로,2019-11-18
1,METEOR,창모 (CHANGMO),Boyhood,4378,419,51,3,18,312720,78.321021,4797,97.658795,94.857463,2014-06-10,남성,솔로,2019-11-29
2,아무노래,지코 (ZICO),아무노래,3667,846,45,8,16,282212,74.402623,4513,91.877036,94.634712,2014-11-07,남성,솔로,2020-01-13
3,"어떻게 이별까지 사랑하겠어, 널 사랑하는 거지",AKMU (악뮤),항해,4469,46,56,0,9,495321,98.569801,4515,91.917752,93.913367,2014-04-07,혼성,그룹,2019-09-25
4,흔들리는 꽃들 속에서 네 샴푸향이 느껴진거야,장범준,멜로가 체질 OST Part 3,4876,24,56,0,24,320125,79.242890,4900,99.755700,93.601857,2014-08-19,남성,솔로,2019-08-23
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
482,사랑 이토록 어려운 말,양다일,낭만닥터 김사부 2 OST Part.5,11,0,1,0,0,7531,12.154213,11,0.223941,3.803023,2015-10-01,남성,솔로,2020-01-27
483,SOON (Feat. BewhY),다이나믹 듀오,SOON,4,0,1,0,0,7495,12.125128,4,0.081433,3.694542,2004-05-17,남성,그룹,2020-11-03
484,나를 기다렸나요,한동근,나를 기다렸나요,15,0,1,0,0,6623,11.397980,15,0.305375,3.633156,2014-09-30,남성,솔로,2019-12-29
485,알았다면,길구봉구,알았다면,9,0,1,0,0,6708,11.470888,9,0.183225,3.569524,2013-04-01,남성,그룹,2020-11-15


In [324]:
# 집계기간
mma_periods = {
    "MMA_2020": ('2019-10-31', '2020-11-19'),
    "MMA_2021": ('2020-10-31', '2021-11-19'),
    "MMA_2022": ('2021-10-31', '2022-11-19'),
    "MMA_2023": ('2022-11-04', '2023-11-01'),
    "MMA_2024": ('2023-11-02', '2024-10-30'),
    "MMA_2025": ('2024-10-31', '2025-11-19')
}

In [341]:
def process_year(df, year):
    # MMA 집계기간 적용
    start, end = mma_periods[f"MMA_{year}"]
    
    # 기간 필터
    df_filtered = df[
        df['release_date'].between(start, end)
    ].copy()
    
    # 아티스트 점수
    artist = df_filtered.groupby('artist').agg(
        artist_score=('total_score', 'sum'),
        song_count=('title', 'count'),
        데뷔=('데뷔', 'first'),
        구성=('구성', 'first'),
        성별=('성별', 'first')
    ).sort_values(by='artist_score', ascending=False).reset_index()
    
    # 앨범 점수
    album = df_filtered.groupby(['album', 'artist']).agg(
        album_score=('total_score', 'sum'),
        track_count=('title', 'count')
    ).sort_values(by='album_score', ascending=False).reset_index()
    
    return df_filtered, artist, album

In [342]:
# 연도별 집계기간 데이터, 아티스트 점수, 앨범 점수
result = {}

for year in range(2020, 2026):
    df_filtered, artist, album = process_year(merged_df[year], year)
    
    result[year] = {
        'filtered': df_filtered,
        'artist': artist,
        'album': album
    }

In [343]:
result[2020]['artist']

,artist,artist_score,song_count,데뷔,구성,성별
0,방탄소년단,560.479754,15,2013-06-13,그룹,남성
1,아이유,454.739810,9,2008-09-18,솔로,여성
2,백현 (BAEKHYUN),195.534299,11,2019-07-10,솔로,남성
3,백예린 (Yerin Baek),173.027344,10,2015-11-30,솔로,여성
4,BLACKPINK,166.604591,7,2016-08-08,그룹,여성
...,...,...,...,...,...,...
159,혁오 (HYUKOH),5.068143,1,2014-09-18,그룹,남성
160,에릭남 (Eric Nam),4.948743,1,2013-01-23,솔로,남성
161,윤딴딴,4.901269,1,2014-02-14,솔로,남성
162,임재현,3.866977,1,2018-09-25,솔로,남성


올해의 아티스트      
올해의 베스트송    
올해의 앨범 
올해의 신인        
베스트 솔로 남자    
베스트 솔로 여자    
베스트 그룹 남자    
베스트 그룹 여자    
TOP10         
밀리언스 TOP10    

In [491]:
# 수상자 예측함수
def predict_awards(year, data):
    predictions = []
    
    df = data['filtered'] # 집계기간 곡별 점수
    artist_df = data['artist'] # 아티스트 점수
    album_df = data['album'] # 앨범 점수
    
    # 1. 올해의 아티스트
    top_artist = artist_df.iloc[0]
    predictions.append({
        '연도': year,
        '수상부문': '올해의 아티스트',
        '수상자': top_artist['artist']
    })
    
    # 2. 올해의 앨범
    top_album = album_df.iloc[0]
    predictions.append({
        '연도': year,
        '수상부문': '올해의 앨범',
        '수상자': top_album['album']
    })
    
    # 3. 올해의 베스트송
    top_song = df.sort_values(by='total_score', ascending=False).iloc[0]
    predictions.append({
        '연도': year,
        '수상부문': '올해의 베스트송',
        '수상자': top_song['title']
    })
    
    # 4. 올해의 신인 (데뷔 기준)
    debut_date = pd.to_datetime(df['데뷔'], errors='coerce')
    rookie_df = df[debut_date.dt.year == year]
    
    rookie_top = rookie_df.sort_values(by='total_score', ascending=False).drop_duplicates('artist').head(2)
    
    for _, row in rookie_top.iterrows():
        predictions.append({
            '연도': year,
            '수상부문': '올해의 신인',
            '수상자': row['artist']
        })
    # 5. 베스트 솔로(여자)
    female_solo_df = (
        artist_df[
            (artist_df['성별'] == '여성') & 
            (artist_df['구성'] == '솔로')
        ]
        .head(1)
    )
    
    predictions.append({
        '연도': year,
        '수상부문': '베스트 솔로 여자',
        '수상자': female_solo_df['artist'].item()
    })
    
    # 6. 베스트 그룹(여자)
    female_group_df = (
        artist_df[
            (artist_df['성별'] == '여성') & 
            (artist_df['구성'] == '그룹')
        ]
        .head(1)
    )
    
    predictions.append({
        '연도': year,
        '수상부문': '베스트 그룹 여자',
        '수상자': female_group_df['artist'].item()
    })
    
    # 7. 베스트 솔로(남자)
    male_solo_df = (
        artist_df[
            (artist_df['성별'] == '남성') & 
            (artist_df['구성'] == '솔로')
        ]
        .head(1)
    )
    
    predictions.append({
        '연도': year,
        '수상부문': '베스트 솔로 남자',
        '수상자': male_solo_df['artist'].item()
    })
    
    # 8. 베스트 그룹(남자)
    male_group_df = (
        artist_df[
            (artist_df['성별'] == '남성') & 
            (artist_df['구성'] == '그룹')
        ]
        .head(1)
    )
    
    predictions.append({
        '연도': year,
        '수상부문': '베스트 그룹 남자',
        '수상자': male_group_df['artist'].item()
    })
  
    
    # 9. TOP10
    top10 = artist_df.sort_values(by='artist_score', ascending=False).head(10)
    for _, row in top10.iterrows():
        predictions.append({
            '연도': year,
            '수상부문': 'TOP10',
            '수상자': row['artist']
        })
    
    # 10. 밀리언스 TOP10
    millions_top10 = album_df.sort_values(by='album_score', ascending=False).head(10)
    for _, row in millions_top10.iterrows():
        predictions.append({
            '연도': year,
            '수상부문': '밀리언스 TOP10',
            '수상자': row['album']
        })
    
    return pd.DataFrame(predictions)

In [492]:
all_preds = {}

for year in result.keys():
    pred = predict_awards(year, result[year])
    all_preds[year] = pred


In [493]:
all_preds[2020]

,연도,수상부문,수상자
0,2020,올해의 아티스트,방탄소년단
1,2020,올해의 앨범,MAP OF THE SOUL : 7
2,2020,올해의 베스트송,Blueming
3,2020,올해의 신인,"싹쓰리 (유두래곤, 린다G, 비룡)"
4,2020,올해의 신인,환불원정대
5,2020,베스트 솔로 여자,아이유
6,2020,베스트 그룹 여자,BLACKPINK
7,2020,베스트 솔로 남자,백현 (BAEKHYUN)
8,2020,베스트 그룹 남자,방탄소년단
9,2020,TOP10,방탄소년단


In [494]:
# 실제 수상 명단
real_award_2025 = pd.read_csv('/Users/js/MMA/Melon_Music_Award/data/MMA_수상내역/2025_MMA_수상내역.csv')
real_award_2024 = pd.read_csv('/Users/js/MMA/Melon_Music_Award/data/MMA_수상내역/2024_MMA_수상내역.csv')
real_award_2023 = pd.read_csv('/Users/js/MMA/Melon_Music_Award/data/MMA_수상내역/2023_MMA_수상내역.csv')
real_award_2022 = pd.read_csv('/Users/js/MMA/Melon_Music_Award/data/MMA_수상내역/2022_MMA_수상내역.csv')
real_award_2021 = pd.read_csv('/Users/js/MMA/Melon_Music_Award/data/MMA_수상내역/2021_MMA_수상내역.csv')
real_award_2020 = pd.read_csv('/Users/js/MMA/Melon_Music_Award/data/MMA_수상내역/2020_MMA_수상내역.csv')

In [495]:
real_award_2020['수상부문'] = real_award_2020['category']
real_award_2021['수상부문'] = real_award_2021['category']
real_award_2022['수상부문'] = real_award_2022['category']
real_award_2023['수상부문'] = real_award_2023['category']
real_award_2024['수상부문'] = real_award_2024['category']
real_award_2025['수상부문'] = real_award_2025['category']

In [496]:
# 수상부문 특이사항
# 밀리언스 Top10 부문은 2023~2025년 MMA에만 존재

# 2020 수상부문 수정
# 아티스트상 > 올해의 아티스트 / 앨범상 > 올해의 앨범 / 베스트송상 > 올해의 베스트송 / 신인상(남자부문),신인상(여자부문) > 올해의 신인 
# 베스트 솔로 남자,여자 / 베스트 그룹 남자,여자 없음

# 2021 수상부문 수정
# 올해의 신인 남자, 올해의 신인 여자 

# 2024 수상부문 수정
# 올해의 신인 남자, 여자 

In [500]:
# 수상부문 명칭 통일 함수
def standardize_award(df, year):
    df = df.copy()
    
    # 1. 기본 명칭 통일
    rename_map = {
        '아티스트상': '올해의 아티스트',
        '앨범상': '올해의 앨범',
        '베스트송상': '올해의 베스트송'
    }
    
    df['수상부문'] = df['수상부문'].replace(rename_map)
    
    # 2. 신인상 통일
    if year in [2020]:
        df['수상부문'] = df['수상부문'].replace({
            '신인상(남자부문)': '올해의 신인',
            '신인상(여자부문)': '올해의 신인'
        })
        
    elif year in [2021, 2024]:
        df['수상부문'] = df['수상부문'].replace({
            '올해의 신인 남자': '올해의 신인',
            '올해의 신인 여자': '올해의 신인'
        })
    
    return df

In [501]:
# 수상부문 수정 적용
real_awards = {
    2020: real_award_2020,
    2021: real_award_2021,
    2022: real_award_2022,
    2023: real_award_2023,
    2024: real_award_2024,
    2025: real_award_2025
}

real_awards_std = {}

for year, df in real_awards.items():
    real_awards_std[year] = standardize_award(df, year)

In [503]:
def make_real_award_value(df):
    df = df.copy()
    
    # 기준 정의
    artist_awards = [
        '올해의 아티스트', '올해의 신인',
        '베스트 솔로 여자', '베스트 솔로 남자',
        '베스트 그룹 여자', '베스트 그룹 남자',
        'TOP10'
    ]
    
    album_awards = ['올해의 앨범', '밀리언스 TOP10']
    song_awards = ['올해의 베스트송']
    
    df['수상자'] = None
    
    df.loc[df['수상부문'].isin(artist_awards), '수상자'] = df['아티스트']
    df.loc[df['수상부문'].isin(album_awards), '수상자'] = df['앨범']
    df.loc[df['수상부문'].isin(song_awards), '수상자'] = df['곡명']
    
    return df[['수상부문', '수상자']]

In [504]:
real_awards_processed = {}

for year in range(2020, 2026):
    df = real_awards_std[year]
    real_awards_processed[year] = make_real_award_value(df)

In [505]:
real_awards_processed[2020]

,수상부문,수상자
0,올해의 아티스트,방탄소년단
1,올해의 앨범,MAP OF THE SOUL : 7
2,올해의 베스트송,Dynamite
3,올해의 신인,CRAVITY
4,올해의 신인,Weeekly (위클리)
5,네티즌 인기상,None
6,핫트렌드상,None
7,댄스 (남자부문),None
8,댄스 (여자부문),None
9,록,None


In [506]:
# 실제 예측할 수상부문
target_awards = [
    '올해의 베스트송',
    '올해의 앨범',
    '올해의 아티스트',
    '올해의 신인',
    '베스트 솔로 여자',
    '베스트 솔로 남자',
    '베스트 그룹 여자',
    '베스트 그룹 남자',
    'TOP10',
    '밀리언스 TOP10'
]

In [507]:
for year in real_awards_processed:
    df = real_awards_processed[year]
    
    real_awards_processed[year] = df[
        df['수상부문'].isin(target_awards)
    ].copy()

In [508]:
for year in real_awards_processed:
    print(year, real_awards_processed[year]['수상부문'].unique())

2020 ['올해의 아티스트' '올해의 앨범' '올해의 베스트송' '올해의 신인' 'TOP10']
2021 ['올해의 앨범' '올해의 아티스트' '올해의 베스트송' '올해의 신인' '베스트 솔로 여자' '베스트 솔로 남자'
 '베스트 그룹 여자' '베스트 그룹 남자' 'TOP10']
2022 ['올해의 앨범' '올해의 아티스트' '올해의 베스트송' '올해의 신인' '베스트 솔로 여자' '베스트 솔로 남자'
 '베스트 그룹 여자' '베스트 그룹 남자' 'TOP10']
2023 ['올해의 앨범' '올해의 아티스트' '올해의 베스트송' '올해의 신인' '베스트 솔로 여자' '베스트 솔로 남자'
 '베스트 그룹 여자' '베스트 그룹 남자' 'TOP10' '밀리언스 TOP10']
2024 ['올해의 앨범' '올해의 아티스트' '올해의 베스트송' '올해의 신인' '베스트 솔로 여자' '베스트 솔로 남자'
 '베스트 그룹 여자' '베스트 그룹 남자' 'TOP10' '밀리언스 TOP10']
2025 ['올해의 앨범' '올해의 아티스트' '올해의 베스트송' '올해의 신인' '베스트 솔로 여자' '베스트 솔로 남자'
 '베스트 그룹 여자' '베스트 그룹 남자' 'TOP10' '밀리언스 TOP10']


In [509]:
def compare_year_strict(year, pred_df, real_df):
    pred_df = pred_df.copy()
    real_df = real_df.copy()
    
    # 1. 해당 연도 실제 수상부문만 가져오기
    valid_awards = real_df['수상부문'].unique()
    
    # 2. 예측도 동일 부문만 남기기
    pred_df = pred_df[pred_df['수상부문'].isin(valid_awards)].copy()
    
    # 3. 문자열 정리
    pred_df['수상자'] = pred_df['수상자'].str.strip()
    real_df['수상자'] = real_df['수상자'].str.strip()
    
    # 4. 비교
    compare = pred_df.merge(
        real_df,
        on=['수상부문', '수상자'],
        how='left',
        indicator=True
    )
    
    compare['정답여부'] = compare['_merge'] == 'both'
    compare['연도'] = year
    
    return compare

In [510]:
all_compare = []
year_summary = []

for year in all_preds.keys():
    
    pred_df = all_preds[year]
    real_df = real_awards_processed[year]
    
    compare = compare_year_strict(year, pred_df, real_df)
    all_compare.append(compare)
    
    correct = compare['정답여부'].sum()
    total = len(compare)
    
    year_summary.append({
        '연도': year,
        '정답수': correct,
        '총개수': total,
        '정확도': correct / total
    })

compare_all_df = pd.concat(all_compare, ignore_index=True)
year_acc_df = pd.DataFrame(year_summary)

In [521]:
# 2021년도 예측 최고, 2022녇도 최악
year_acc_df

,연도,정답수,총개수,정확도
0,2020,9,15,0.600000
1,2021,15,19,0.789474
2,2022,8,19,0.421053
3,2023,19,29,0.655172
4,2024,20,29,0.689655
5,2025,19,29,0.655172


In [523]:
compare_all_df[compare_all_df['연도']==2021]

,연도,수상부문,수상자,_merge,정답여부
15,2021,올해의 아티스트,아이유,both,True
16,2021,올해의 앨범,IU 5th Album 'LILAC',both,True
17,2021,올해의 베스트송,Celebrity,left_only,False
18,2021,올해의 신인,이무진,both,True
19,2021,올해의 신인,MSG워너비(M.O.M),left_only,False
20,2021,베스트 솔로 여자,아이유,both,True
21,2021,베스트 그룹 여자,aespa,both,True
22,2021,베스트 솔로 남자,이무진,left_only,False
23,2021,베스트 그룹 남자,방탄소년단,both,True
24,2021,TOP10,아이유,both,True


In [525]:
compare_all_df[compare_all_df['연도']==2022]

,연도,수상부문,수상자,_merge,정답여부
34,2022,올해의 아티스트,BE'O (비오),left_only,False
35,2022,올해의 앨범,NewJeans 1st EP 'New Jeans',left_only,False
36,2022,올해의 베스트송,취중고백,left_only,False
37,2022,올해의 신인,WSG워너비 (가야G),left_only,False
38,2022,올해의 신인,YENA (최예나),left_only,False
39,2022,베스트 솔로 여자,아이유,both,True
40,2022,베스트 그룹 여자,IVE (아이브),both,True
41,2022,베스트 솔로 남자,BE'O (비오),left_only,False
42,2022,베스트 그룹 남자,멜로망스,left_only,False
43,2022,TOP10,BE'O (비오),both,True


In [526]:
compare_all_df.to_csv('전체MMA예측결과_v1.csv', index=False, encoding='utf-8-sig')